In [0]:
from datetime import date, timedelta
from pyspark.sql import functions as F
from pyspark.sql.types import *

In [0]:
%run ../../utils/reader_utils

In [0]:
%run ../../utils/writer_utils

In [0]:
%run ../../utils/transform_utils

In [0]:
# Configs
configs = dict(dbutils.notebook.entry_point.getCurrentBindings())
today = date.today()

ENV = configs.get("env", "dev")
INITIAL_RUN = configs.get('initial_run', 'False').lower() == "true"
START_DATE = configs.get("start_date") or today - timedelta(days=1)
END_DATE = configs.get("end_date") or today + timedelta(days=1)
CATALOG = f"sl_{ENV}"

print(ENV, INITIAL_RUN, START_DATE, END_DATE, CATALOG)

In [0]:
# Constants
SOURCE_TABLE_CONF = {
    "table": "pyspark_scd2_customers",
    "schema": "silver",
    "timestamp_col": "_insert_update_ts"
}

DIM_CUSTOMER_TABLE_CONF = {
    "table": "pyspark_dim_customer",
    "schema": "gold",
    "merge_keys": ['customer_id']
}

FACT_ORDER_FULFILLMENT_TABLE_CONF = {
    "table": "pyspark_fact_order_fulfillment",
    "schema": "gold",
    "merge_keys": ['customer_sk']
}

In [0]:
spark.sql(f"USE CATALOG {CATALOG}")

In [0]:
customers_df = (read_delta_table(SOURCE_TABLE_CONF, START_DATE, END_DATE)
                .filter((F.col("_is_delete"))))

In [0]:
customer_sks = (spark.table(f"""{DIM_CUSTOMER_TABLE_CONF.get("schema")}
                           .{DIM_CUSTOMER_TABLE_CONF.get("table")} 
                           """)
                .join(customers_df, "customer_id", "inner")
                .select("customer_sk"))

In [0]:
delete_in_delta_table(customer_sks, FACT_ORDER_FULFILLMENT_TABLE_CONF)

In [0]:
delete_in_delta_table(customers_df, DIM_CUSTOMER_TABLE_CONF)